# Explore the Extracted Lange Laufnacht Dataset

This notebook loads all currently configured event years from `data/`, keeps the default target events, and caches the extracted table as `data/extracted/results.parquet` for faster reloads.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "lln_stats").exists():
            return path
    raise FileNotFoundError("Could not find the lln-stats repository root")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

REPO_ROOT

In [ ]:
from lln_stats.extract import extract_all
from lln_stats.years import YEAR_SOURCES

CACHE_PATH = REPO_ROOT / "data" / "extracted" / "results.parquet"

if CACHE_PATH.exists():
    df = pd.read_parquet(CACHE_PATH)
else:
    df = extract_all(root=REPO_ROOT)
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(CACHE_PATH, index=False)

df = df.assign(
    time=pd.to_timedelta(df["time_seconds"], unit="s"),
    age=df["event_year"] - df["year_of_birth"],
)

{
    "rows": len(df),
    "columns": len(df.columns),
    "years_loaded": sorted(YEAR_SOURCES),
    "cache_path": str(CACHE_PATH.relative_to(REPO_ROOT)),
}

In [ ]:
df.head(20)

In [ ]:
overview = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "null": df.isna().sum(),
        "unique": df.nunique(dropna=True),
    }
)
overview

In [ ]:
pd.crosstab(
    index=[df["event_year"], df["event"]],
    columns=df["gender"],
    margins=True,
)

In [ ]:
finished = df[df["status"].eq("finished")].copy()

annual_bests = (
    finished.sort_values("time_seconds")
    .groupby(["event_year", "event", "gender"], dropna=False)
    .first()
    .reset_index()
)

annual_bests[
    [
        "event_year",
        "event",
        "gender",
        "athlete_name",
        "club",
        "nationality",
        "result_raw",
        "time_seconds",
    ]
].sort_values(["event", "gender", "event_year"])

In [ ]:
all_time_top10 = (
    finished.sort_values("time_seconds")
    .groupby(["event", "gender"], dropna=False)
    .head(10)
)

all_time_top10[
    [
        "event",
        "gender",
        "athlete_name",
        "event_year",
        "club",
        "nationality",
        "result_raw",
        "time_seconds",
    ]
].reset_index(drop=True)

In [ ]:
ATHLETE_QUERY = ""

if ATHLETE_QUERY:
    matches = df[df["athlete_name"].str.contains(ATHLETE_QUERY, case=False, na=False)]
    matches.sort_values(["athlete_name", "event_year", "event", "time_seconds"])
else:
    "Set ATHLETE_QUERY to part of a name, then rerun this cell."

matches


In [ ]:
EVENT = "1500m"
GENDER = "M"

subset = finished[(finished["event"].eq(EVENT)) & (finished["gender"].eq(GENDER))]
subset[
    [
        "event_year",
        "athlete_name",
        "club",
        "nationality",
        "result_raw",
        "time_seconds",
        "heat",
        "rank_within_heat",
    ]
].sort_values("time_seconds").head(25)